# 03 - Linear Regression\n\nBaseline em Spark MLlib com split temporal, encoding de categoricas e persistencia de metricas.

In [ ]:
import time\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport seaborn as sns\nfrom pyspark.sql import SparkSession, functions as F\nfrom pyspark.ml import Pipeline\nfrom pyspark.ml.evaluation import RegressionEvaluator\nfrom pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler\nfrom pyspark.ml.regression import LinearRegression\n\nSEED = 42\nSPLIT_DATE = '2023-06-01'\nSILVER_PATH = '/data/silver/trips_silver'\nMODEL_OUTPUT = '/models/linear_regression'\nRESULTS_PATH = Path('/results/model_comparison.csv')\nRESIDUAL_PLOT = Path('/results/residual_analysis_linear_regression.png')\nTARGET_COL = 'base_passenger_fare'\n\nNUMERIC_COLS = [\n    'trip_miles', 'trip_time', 'wait_time_sec', 'speed_mph', 'pickup_hour',\n    'pickup_dow', 'pickup_month_num', 'is_weekend', 'is_rush_hour',\n    'is_late_night', 'pickup_airport', 'dropoff_airport', 'same_borough',\n    'shared_req', 'wav_req'\n]\nCATEGORICAL_COLS = ['hvfhs_license_num', 'pu_borough', 'do_borough']\n\nspark = (SparkSession.builder\n    .appName('nyc-rideshare-linear-regression')\n    .master('spark://spark-master:7077')\n    .config('spark.executor.memory', '3g')\n    .config('spark.driver.memory', '4g')\n    .config('spark.sql.shuffle.partitions', '200')\n    .getOrCreate())\n\nspark.sparkContext.setLogLevel('WARN')\nsns.set_theme(style='whitegrid')

In [ ]:
silver_df = spark.read.parquet(SILVER_PATH)
silver_df.createOrReplaceTempView('trips_silver')

# Split temporal via Spark SQL puro (regra do projeto: data prep em SQL, ML pipeline em Python).
train_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime < TIMESTAMP '{SPLIT_DATE}'
""").cache()
test_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime >= TIMESTAMP '{SPLIT_DATE}'
""").cache()

train_rows = train_df.count()
test_rows = test_df.count()
if train_rows == 0 or test_rows == 0:
    raise ValueError('O split temporal exige dados antes e depois de 2023-06-01. Um unico mes como 2023-08 nao basta para treinar e avaliar.')

train_rows, test_rows

In [ ]:
indexers = [\n    StringIndexer(inputCol=col_name, outputCol=f'{col_name}_idx', handleInvalid='keep')\n    for col_name in CATEGORICAL_COLS\n]\nencoders = [\n    OneHotEncoder(inputCol=f'{col_name}_idx', outputCol=f'{col_name}_ohe')\n    for col_name in CATEGORICAL_COLS\n]\nnumeric_assembler = VectorAssembler(inputCols=NUMERIC_COLS, outputCol='numeric_features')\nscaler = StandardScaler(inputCol='numeric_features', outputCol='scaled_numeric_features', withMean=False, withStd=True)\nfeature_assembler = VectorAssembler(\n    inputCols=['scaled_numeric_features'] + [f'{col_name}_ohe' for col_name in CATEGORICAL_COLS],\n    outputCol='features'\n)\nlr = LinearRegression(\n    labelCol=TARGET_COL,\n    featuresCol='features',\n    predictionCol='prediction',\n    regParam=0.0,\n    elasticNetParam=0.0\n)\n\npipeline = Pipeline(stages=indexers + encoders + [numeric_assembler, scaler, feature_assembler, lr])

In [ ]:
start_time = time.perf_counter()\nlr_model = pipeline.fit(train_df)\ntrain_seconds = time.perf_counter() - start_time\n\nlr_model.write().overwrite().save(MODEL_OUTPUT)\nprint(f'Modelo salvo em {MODEL_OUTPUT} | treino: {train_seconds:.2f}s')

In [ ]:
predictions = lr_model.transform(test_df).cache()\nevaluators = {\n    'rmse': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='rmse'),\n    'mae': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='mae'),\n    'r2': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='r2')\n}\nmetrics = {name: evaluator.evaluate(predictions) for name, evaluator in evaluators.items()}\nmetrics

In [ ]:
predictions.createOrReplaceTempView('predictions_lr')\n\nspark.sql("""\nSELECT\n    pu_borough,\n    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n    COUNT(*) AS rows\nFROM predictions_lr\nGROUP BY 1\nORDER BY rmse DESC\n""").show(truncate=False)\n\nspark.sql("""\nSELECT\n    CASE\n        WHEN trip_miles < 2 THEN '0-2 mi'\n        WHEN trip_miles < 5 THEN '2-5 mi'\n        WHEN trip_miles < 10 THEN '5-10 mi'\n        ELSE '10+ mi'\n    END AS distance_bucket,\n    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n    COUNT(*) AS rows\nFROM predictions_lr\nGROUP BY 1\nORDER BY distance_bucket\n""").show(truncate=False)

In [ ]:
residual_pdf = (predictions\n    .select('prediction', TARGET_COL)\n    .sample(False, 0.02, SEED)\n    .toPandas())\nresidual_pdf['residual'] = residual_pdf[TARGET_COL] - residual_pdf['prediction']\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\nsns.histplot(residual_pdf['residual'], bins=50, ax=axes[0])\naxes[0].set_title('Distribuicao dos residuos - LR')\nsns.scatterplot(data=residual_pdf, x='prediction', y='residual', s=10, alpha=0.3, ax=axes[1])\naxes[1].axhline(0, color='black', linestyle='--', linewidth=1)\naxes[1].set_title('Residuo vs predito - LR')\nplt.tight_layout()\nplt.savefig(RESIDUAL_PLOT, dpi=150, bbox_inches='tight')\nplt.show()

In [ ]:
results_df = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame(columns=['model', 'rmse', 'mae', 'r2', 'train_seconds', 'notes'])\nresults_df = results_df[results_df['model'] != 'linear_regression']\nresults_df = pd.concat([\n    results_df,\n    pd.DataFrame([{\n        'model': 'linear_regression',\n        'rmse': metrics['rmse'],\n        'mae': metrics['mae'],\n        'r2': metrics['r2'],\n        'train_seconds': train_seconds,\n        'notes': 'baseline com StandardScaler e one-hot encoding'\n    }])\n], ignore_index=True)\nresults_df.to_csv(RESULTS_PATH, index=False)\nresults_df

In [ ]:
spark.stop()